# 01_build_cohort (NACC) — Updated

Builds a participant-level cohort from `data/investigator_nacc72.csv` and assigns 5-year outcome labels using dictionary semantics:
- AD: `NACCALZP`
- Vascular (UDS v3.x): `CVD`, `CVDIF`
- PD/LBD: `NACCLBDP`, `NACCLBDE`

Outputs: `output/cohort.csv`


In [1]:
import os
import re
import numpy as np
import pandas as pd

DATA_DIR = os.path.join("..", "data")
OUT_DIR  = os.path.join("..", "output")
os.makedirs(OUT_DIR, exist_ok=True)

NACC_MAIN_PATH = os.path.join(DATA_DIR, "investigator_nacc72.csv")
NACC_CSF_PATH  = os.path.join(DATA_DIR, "investigator_fcsf_nacc72.csv")
DICT_PATH      = os.path.join(DATA_DIR, "nacc_variable_dictionary_consolidated.xlsx")

HORIZON_YEARS = 5

print("Main:", NACC_MAIN_PATH)
print("CSF :", NACC_CSF_PATH)
print("Dict:", DICT_PATH)


Main: ..\data\investigator_nacc72.csv
CSF : ..\data\investigator_fcsf_nacc72.csv
Dict: ..\data\nacc_variable_dictionary_consolidated.xlsx


## Dictionary (optional) + cleaning helpers

In [2]:
import os
import re
import numpy as np
import pandas as pd

STANDARD_MISSING = {-4, 9}


def clean_coded(series: pd.Series, extra_missing=None):
    s = pd.to_numeric(series, errors="coerce")
    miss = set(STANDARD_MISSING)
    if extra_missing:
        miss |= set(extra_missing)
    return s.mask(s.isin(list(miss)), np.nan)


def _normalize_allowable_text(s: str) -> str:
    return s.replace(" - ", " -").replace("- ", "-")

def parse_code_labels(allowable_codes: str):
    """Parse strings like '0 = ... 1 = ... 9 = Missing -4 = Not available' into [(code,label), ...]."""
    if not isinstance(allowable_codes, str) or not allowable_codes.strip():
        return []
    txt = _normalize_allowable_text(allowable_codes)
    parts = re.split(r'(?=(?:^|\s)(-?\d+)\s*=\s*)', txt)

    out = []
    if len(parts) < 3:
        return out

    i = 1
    while i < len(parts) - 1:
        try:
            code = int(parts[i])
        except Exception:
            i += 2
            continue
        rest = parts[i + 1]
        label = re.sub(r'\s+', ' ', str(rest)).strip().strip(" -;")
        out.append((code, label))
        i += 2
    return out

def _pick_columns(df: pd.DataFrame):
    """Find best guess for variable-name column and allowable-codes column."""
    cols = {c.lower().strip(): c for c in df.columns}

    var_candidates = [
        "variable", "varname", "name", "field", "field_name", "variable_name"
    ]
    allow_candidates = [
        "allowable_codes", "allowable code", "allowed_values", "allowed values",
        "codes", "values", "allowablecodes", "allowedvalues"
    ]

    var_col = next((cols[c] for c in var_candidates if c in cols), None)
    allow_col = next((cols[c] for c in allow_candidates if c in cols), None)
    return var_col, allow_col

def build_missing_code_map_debug(dict_xlsx_path: str):
    """
    Debug-friendly version:
    - Raises clear errors if file/sheets/columns aren't found.
    - Searches across all sheets for columns if needed.
    """
    if not os.path.exists(dict_xlsx_path):
        raise FileNotFoundError(
            f"Dictionary file not found at: {dict_xlsx_path}\n"
            f"Make sure it is placed in ./data/ and the name matches exactly."
        )

    xl = pd.ExcelFile(dict_xlsx_path)
    print("✅ Dictionary file opened.")
    print("Sheets:", xl.sheet_names)

    # 1) Try preferred sheet names first
    preferred = {"dictionary", "data dictionary", "dict"}
    sheet_order = []
    for s in xl.sheet_names:
        if s.lower().strip() in preferred:
            sheet_order.insert(0, s)
        else:
            sheet_order.append(s)

    chosen_sheet = None
    chosen_var_col = None
    chosen_allow_col = None
    chosen_df = None

    # 2) Find first sheet that contains suitable columns
    for s in sheet_order:
        tmp = pd.read_excel(dict_xlsx_path, sheet_name=s)
        var_col, allow_col = _pick_columns(tmp)
        if var_col is not None and allow_col is not None:
            chosen_sheet = s
            chosen_var_col = var_col
            chosen_allow_col = allow_col
            chosen_df = tmp
            break

    if chosen_sheet is None:
        # print columns for debugging
        for s in xl.sheet_names:
            tmp = pd.read_excel(dict_xlsx_path, sheet_name=s)
            print(f"\nSheet '{s}' columns:\n{list(tmp.columns)}")
        raise ValueError(
            "Could not find required columns in ANY sheet.\n"
            "Need columns like 'variable' and 'allowable_codes' (names may differ). "
            "See printed columns above."
        )

    print(f"\n✅ Using sheet: '{chosen_sheet}'")
    print(f"   var column:   {chosen_var_col}")
    print(f"   codes column: {chosen_allow_col}")

    d = chosen_df

    missing_map = {}
    label_map = {}

    for _, r in d.iterrows():
        var = r.get(chosen_var_col, None)
        allow = r.get(chosen_allow_col, "")

        if not isinstance(var, str) or not var.strip():
            continue

        pairs = parse_code_labels(allow)
        code_to_label = {c: lbl for c, lbl in pairs}
        label_map[var] = code_to_label

        miss = set()
        for c, lbl in pairs:
            l = str(lbl).lower()
            if ("missing" in l) or ("not available" in l) or ("not applicable" in l) or ("unknown" in l):
                miss.add(c)
        missing_map[var] = miss

    meta = (chosen_sheet, chosen_var_col, chosen_allow_col, len(label_map))
    return missing_map, label_map, meta

# --- Run debug load ---
missing_map, label_map, meta = build_missing_code_map_debug(DICT_PATH)
print("\nDictionary read OK:", meta)

# Quick sanity checks
for v in ["COGSTAT", "CVDIF", "NACCALZP", "NACCLBDP", "NACCCUDSD"]:
    ex = list(label_map.get(v, {}).items())[:8]
    print(f"{v} example codes:", ex if ex else "(not found in dictionary)")


✅ Dictionary file opened.
Sheets: ['Dictionary', 'RawSections']

✅ Using sheet: 'Dictionary'
   var column:   variable
   codes column: allowable_codes

Dictionary read OK: ('Dictionary', 'variable', 'allowable_codes', 1044)
COGSTAT example codes: [(0, '0 = Clinician unable to render opinion Not applic Not applic (Form C1 or C2)'), (1, '1 = Better than normal for age TI question no. 13a (Form C2T)'), (2, '2 = Normal for age'), (3, '3 = One or two test scores abnormal TF question no. Not applic Not applic 13a (Form C2T)'), (4, '4 = Three or more scores are abnormal or lower than expected'), (9, '9 = Missing'), (-4, '4 = Not available: UDS form submitted did not collect data in this way, or a skip pattern precludes response to this question')]
CVDIF example codes: [(1, '1 = Primary TI question no. 15a'), (2, '2 = Contributing TF question no. Not applic 15a'), (3, '3 = Non-contributing'), (7, '7 = Cognitively impaired but no VBI diagnosis'), (8, '8 = Diagnosis of normal cognition'), (-4, 

## Load main file + visit_date

In [3]:
df = pd.read_csv(NACC_MAIN_PATH, low_memory=False)
df["NACCID"] = df["NACCID"].astype(str)

def make_visit_date(df_: pd.DataFrame) -> pd.Series:
    y = pd.to_numeric(df_["VISITYR"], errors="coerce")
    m = pd.to_numeric(df_["VISITMO"], errors="coerce").fillna(6).astype(int).clip(1, 12)
    d = pd.to_numeric(df_["VISITDAY"], errors="coerce").fillna(15).astype(int).clip(1, 28)
    return pd.to_datetime(dict(year=y, month=m, day=d), errors="coerce")

df["visit_date"] = make_visit_date(df)
df = df.dropna(subset=["NACCID", "visit_date"]).copy()
df = df.sort_values(["NACCID", "visit_date"]).reset_index(drop=True)

if "FORMVER" in df.columns:
    df["FORMVER"] = pd.to_numeric(df["FORMVER"], errors="coerce")

print("Rows:", len(df), "Participants:", df["NACCID"].nunique())
df[["NACCID","visit_date","FORMVER"]].head()


Rows: 207454 Participants: 55268


,NACCID,visit_date,FORMVER
0,NACC000011,2006-04-17,1.0
1,NACC000011,2007-06-18,1.0
2,NACC000011,2008-06-03,2.0
3,NACC000011,2009-08-03,2.0
4,NACC000034,2015-07-16,3.0


## Optional: restrict to UDS v3 / 3.2 only

In [4]:
USE_ONLY_V3 = True  # set True to keep only FORMVER in {3, 3.2}

if USE_ONLY_V3 and "FORMVER" in df.columns:
    before = len(df)
    df = df[df["FORMVER"].isin([3, 3.2])].copy()
    df = df.sort_values(["NACCID","visit_date"]).reset_index(drop=True)
    print("Filtered to FORMVER in {3,3.2}: rows", before, "->", len(df))
else:
    print("No FORMVER filtering applied.")


Filtered to FORMVER in {3,3.2}: rows 207454 -> 103543


## Clean key status + etiology variables

In [5]:
STATUS_VARS = ["NACCCUDSD", "DEMENTED", "NORMCOG", "NACCTMCI", "COGSTAT"]
AD_VARS   = ["NACCALZD", "NACCALZP"]
VBI_VARS  = ["CVD", "CVDIF"]
LBD_VARS  = ["NACCLBDE", "NACCLBDP"]
OLD_VASC_VARS = ["VASC", "VASCIF", "VASCPS", "VASCPSIF", "STROKE", "STROKIF"]

for v in STATUS_VARS + AD_VARS + VBI_VARS + LBD_VARS + OLD_VASC_VARS:
    if v in df.columns:
        extra = missing_map.get(v, set()) if missing_map else set()
        df[v] = clean_coded(df[v], extra_missing=extra)

# COGSTAT: 0 = clinician unable to render opinion -> treat as missing
if "COGSTAT" in df.columns:
    df.loc[df["COGSTAT"] == 0, "COGSTAT"] = np.nan

df[[c for c in ["NACCCUDSD","DEMENTED","NORMCOG","NACCTMCI","COGSTAT",
                "NACCALZD","NACCALZP","CVD","CVDIF","NACCLBDE","NACCLBDP"]
    if c in df.columns]].head()


,DEMENTED,NORMCOG,NACCTMCI,COGSTAT,NACCALZD,NACCALZP,CVD,CVDIF,NACCLBDE,NACCLBDP
0,0,0,1,2.0,1,1,0,7,0,7
1,0,0,3,3.0,1,1,0,7,0,7
2,0,0,1,2.0,1,1,0,7,0,7
3,0,0,2,4.0,1,1,0,7,0,7
4,0,0,2,4.0,1,1,0,7,0,7


## Baseline selection (CN or MCI; not demented)

In [6]:
def first_row_per_group(df_: pd.DataFrame, group_col: str, sort_col: str) -> pd.DataFrame:
    return df_.sort_values([group_col, sort_col]).groupby(group_col, as_index=False).first()

if "NACCCUDSD" in df.columns:
    cnmci_mask = df["NACCCUDSD"].isin([1, 3])
else:
    cn_mask  = (df["NORMCOG"] == 1) if "NORMCOG" in df.columns else False
    mci_mask = df["NACCTMCI"].isin([1,2,3,4]) if "NACCTMCI" in df.columns else False
    cnmci_mask = cn_mask | mci_mask

no_dem_mask = (df["DEMENTED"] == 0) if "DEMENTED" in df.columns else True

baseline_candidates = df[cnmci_mask & no_dem_mask].copy()
baseline = first_row_per_group(baseline_candidates, "NACCID", "visit_date").copy()
baseline = baseline.rename(columns={"visit_date": "baseline_date"})

print("Baseline participants:", len(baseline))
baseline[["NACCID","baseline_date"] + [c for c in ["NACCCUDSD","DEMENTED"] if c in baseline.columns]].head()


C:\Users\branjan\AppData\Local\Temp\ipykernel_27812\233066364.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return df_.sort_values([group_col, sort_col]).groupby(group_col, as_index=False).first()


Baseline participants: 23707


,NACCID,baseline_date,DEMENTED
0,NACC000034,2015-07-16,0
1,NACC000073,2016-05-20,0
2,NACC000144,2015-09-08,0
3,NACC000162,2015-08-06,0
4,NACC000176,2021-02-15,0


## 5-year window, censoring, first dementia visit

In [7]:
def years_between(a: pd.Series, b: pd.Series) -> pd.Series:
    return (b - a).dt.days / 365.25

keep_ids = set(baseline["NACCID"])
df2 = df[df["NACCID"].isin(keep_ids)].merge(
    baseline[["NACCID","baseline_date"]], on="NACCID", how="left"
).copy()

df2["t_years"] = years_between(df2["baseline_date"], df2["visit_date"])
post = df2[df2["t_years"] > 0].copy()

max_follow = post.groupby("NACCID")["t_years"].max().rename("max_follow_years")
cohort = baseline.merge(max_follow, on="NACCID", how="left")
cohort["max_follow_years"] = cohort["max_follow_years"].fillna(0)
cohort["has_5y_followup"] = cohort["max_follow_years"] >= HORIZON_YEARS

window = post[post["t_years"] <= HORIZON_YEARS].copy()

if "NACCCUDSD" in window.columns:
    dem_events = window[window["NACCCUDSD"] == 4].copy()
else:
    dem_events = window[window["DEMENTED"] == 1].copy()

first_dem = dem_events.sort_values(["NACCID","visit_date"]).groupby("NACCID", as_index=False).first()
first_dem = first_dem.rename(columns={"visit_date":"dementia_date"})

print("Participants with dementia within 5y:", first_dem["NACCID"].nunique())
first_dem[["NACCID","dementia_date"] + [c for c in ["FORMVER","NACCCUDSD","DEMENTED"] if c in first_dem.columns]].head()


Participants with dementia within 5y: 1819


C:\Users\branjan\AppData\Local\Temp\ipykernel_27812\3051470098.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  first_dem = dem_events.sort_values(["NACCID","visit_date"]).groupby("NACCID", as_index=False).first()


,NACCID,dementia_date,FORMVER,DEMENTED
0,NACC000162,2017-11-10,3.0,1
1,NACC000333,2024-07-24,3.0,1
2,NACC000626,2024-06-10,3.0,1
3,NACC000882,2019-07-23,3.0,1
4,NACC001456,2024-03-08,3.0,1


## Etiology labeling (PRIMARY only) — updated from your dictionary screenshots

In [8]:
MISSING = {-4, 9}

INCLUDE_CONTRIBUTING = True  # <-- set False to go back to primary-only

def _num(x):
    x = pd.to_numeric(x, errors="coerce")
    if pd.isna(x) or x in MISSING:
        return np.nan
    return x

def label_subtype_primary_or_contrib(row: pd.Series) -> str:
    ok = {1, 2} if INCLUDE_CONTRIBUTING else {1}

    # AD: NACCALZP (1=primary, 2=contributing)
    alzp = _num(row.get("NACCALZP"))
    ad_ok = (alzp in ok)

    # PD/LBD: NACCLBDP (1=primary, 2=contributing)
    lbdp = _num(row.get("NACCLBDP"))
    lbd_ok = (lbdp in ok)

    # Vascular: v3.x uses CVD + CVDIF; older uses VASC* variables
    formver = _num(row.get("FORMVER"))
    vascular_ok = False

    if (not np.isnan(formver)) and (formver >= 3):
        cvd  = _num(row.get("CVD"))     # 1 yes, 0 no
        cvdif = _num(row.get("CVDIF"))  # 1 primary, 2 contributing, 3 non, 7/8 special
        vascular_ok = (cvd == 1) and (cvdif in ok)
    else:
        vasc = _num(row.get("VASC"))
        vascif = _num(row.get("VASCIF"))
        vascpsif = _num(row.get("VASCPSIF"))
        vascular_ok = (vasc == 1) and ((vascif in ok) or (vascpsif in ok))

    prim = {"AD": ad_ok, "Vascular": vascular_ok, "PD_LBD": lbd_ok}
    chosen = [k for k, v in prim.items() if bool(v)]
    if len(chosen) == 1:
        return chosen[0]
    return "MixedOrOther"

if len(first_dem) > 0:
    first_dem["etiology_primary"] = first_dem.apply(label_subtype_primary_or_contrib, axis=1)
else:
    first_dem["etiology_primary"] = pd.Series(dtype=str)

first_dem["etiology_primary"].value_counts(dropna=False)


etiology_primary
AD              1202
MixedOrOther     438
PD_LBD           125
Vascular          54
Name: count, dtype: int64

## Final label assignment (4-class + Mixed/Censored helpers)

In [9]:
cohort = cohort.merge(
    first_dem[["NACCID","dementia_date","etiology_primary"]],
    on="NACCID", how="left"
)
cohort["has_dementia_5y"] = cohort["dementia_date"].notna()

def assign_label(r):
    if r["has_dementia_5y"]:
        if r["etiology_primary"] in {"AD","Vascular","PD_LBD"}:
            return r["etiology_primary"]
        return "MixedOrOther"
    if r["has_5y_followup"]:
        return "NoDementia_5y"
    return "Censored"

cohort["label"] = cohort.apply(assign_label, axis=1)
cohort["label"].value_counts(dropna=False)


label
Censored         15681
NoDementia_5y     6207
AD                1202
MixedOrOther       438
PD_LBD             125
Vascular            54
Name: count, dtype: int64

## Build baseline feature table (starter) + optional CSF merge

In [ ]:
STARTER_FEATURES = [
    "NACCID","baseline_date",
    "NACCAGE","SEX","RACE","HISPANIC","EDUC",
    "NACCCUDSD","COGSTAT","NORMCOG","NACCTMCI",
    "DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA","APNEA","DEP2YRS",
    "NACCMMSE","NACCMOCA","TRAILA","TRAILB","ANIMALS","BOSTON",
    "CDRSUM","CDRGLOB",
]

cols = [c for c in STARTER_FEATURES if c in baseline.columns]
X = baseline[cols].copy()

X = X.merge(
    cohort[["NACCID","baseline_date","label","has_5y_followup","max_follow_years"]],
    on=["NACCID","baseline_date"], how="left"
)

NUM_VARS = ["NACCAGE","EDUC","NACCMMSE","NACCMOCA","TRAILA","TRAILB","CDRSUM","CDRGLOB"]
for v in NUM_VARS:
    if v in X.columns:
        extra = missing_map.get(v, set()) if missing_map else set()
        X[v] = clean_coded(X[v], extra_missing=extra)

for v in ["TRAILB","TRAILA","NACCMMSE","NACCMOCA","CDRSUM"]:
    if v in X.columns:
        X[f"{v}_missing"] = X[v].isna().astype(int)

X.head()


In [ ]:
# Optional: merge nearest CSF biomarkers to baseline
if not os.path.exists(NACC_CSF_PATH):
    print("CSF file not found; skipping.")
else:
    csf = pd.read_csv(NACC_CSF_PATH, low_memory=False)
    csf["NACCID"] = csf["NACCID"].astype(str)

    for c in ["CSFABETA","CSFPTAU","CSFTTAU"]:
        if c in csf.columns:
            csf[c] = pd.to_numeric(csf[c], errors="coerce")

    def make_csf_date(df_: pd.DataFrame) -> pd.Series:
        y = pd.to_numeric(df_.get("CSFLPYR"), errors="coerce")
        m = pd.to_numeric(df_.get("CSFLPMO"), errors="coerce").fillna(6).astype(int).clip(1,12)
        d = pd.to_numeric(df_.get("CSFLPDY"), errors="coerce").fillna(15).astype(int).clip(1,28)
        return pd.to_datetime(dict(year=y, month=m, day=d), errors="coerce")

    csf["csf_date"] = make_csf_date(csf)

    csf = csf.merge(X[["NACCID","baseline_date"]], on="NACCID", how="inner")
    csf = csf.dropna(subset=["csf_date","baseline_date"]).copy()
    csf["abs_days_from_baseline"] = (csf["csf_date"] - csf["baseline_date"]).abs().dt.days

    csf_nearest = csf.sort_values(["NACCID","abs_days_from_baseline"]).groupby("NACCID", as_index=False).first()

    merge_cols = ["NACCID","CSFABETA","CSFPTAU","CSFTTAU","abs_days_from_baseline"]
    merge_cols = [c for c in merge_cols if c in csf_nearest.columns]
    X = X.merge(csf_nearest[merge_cols], on="NACCID", how="left")

    print("Merged CSF. Rows:", len(X))
    if "CSFABETA" in X.columns:
        print("CSFABETA non-null:", X["CSFABETA"].notna().sum())

X.head()


## Save cohort.csv

In [ ]:
out_path = os.path.join(OUT_DIR, "cohort.csv")
X.to_csv(out_path, index=False)
print("Saved:", out_path)
print(X["label"].value_counts(dropna=False))
